# 03 · Feature Engineering — IEEE-CIS Fraud Detection

Este notebook parte de `data/processed/train_clean.parquet` y `test_clean.parquet` (generados en
`02_limpieza.ipynb`) y añade features derivadas. Todas las transformaciones se ajustan **solo con
train** y se aplican igual a test.

## Investigación previa (WebSearch) — técnicas específicas para este dataset

Antes de escribir código se buscó en la web qué técnicas de feature engineering usan las
soluciones públicas de Kaggle para *IEEE-CIS Fraud Detection*. Resumen de lo encontrado y qué se
implementó a partir de cada fuente:

1. **Feature "UID" (`card1` + `addr1` + `D1n`)** — descrito en el hilo oficial de discusión de la
   competencia y replicado en varias soluciones públicas: *"IEEE-CIS Fraud Detection" discussion
   #101203* (Kaggle) y el resumen de Priyank Mishra, *"A realistic approach to Kaggle's IEEE-CIS
   Fraud Detection Challenge"* (Medium). La idea (atribuida a Chris Deotte, uno de los ganadores de
   la competencia): `D1` es aproximadamente "días desde que este cliente empezó a operar", así que
   `D1n = día_de_la_transacción - D1` se mantiene casi constante para transacciones del mismo
   cliente a lo largo del tiempo. Combinando `card1 + addr1 + D1n` se obtiene un pseudo-identificador
   de cliente (`UID`) que no existe explícitamente en los datos, descrito como "magic feature" por
   la mejora que produce en los modelos. Aquí se implementa el `UID` y se usa como una columna más
   de agrupación para frequency encoding.
2. **Reducción de las columnas `V1..V339` por correlación (no PCA)** — de *"Reviewing IEEE-CIS
   Fraud Detection" (Karthik Billa, Medium, top-2 solution write-up)* y del artículo *"IEEE-CIS
   Fraud Detection - Top 5% Solution" (Arun Mohan, Towards Data Science)*: ambas fuentes reportan
   que agrupar las columnas `V*` por patrón de NaN y eliminar las que están muy correlacionadas
   entre sí (Pearson) funciona mejor que aplicar PCA sobre esos bloques. Aquí se implementa una
   versión simplificada: se calcula la matriz de correlación de las columnas `V*` **solo en train**
   y se elimina una de cada par con `|corr| > 0.90`.
3. **Agregaciones (frequency encoding + media de `TransactionAmt`) por `card1`, `card2`, `addr1`**
   — mencionado en el mismo top-5% solution write-up ("Feature Aggregations and Frequency
   Encodings" fueron la mayor fuente de nuevas features en esa solución, pasando de ~159 a 312
   columnas) y en el blog técnico de NVIDIA *"Leveraging Machine Learning to Detect Fraud: Tips to
   Developing a Winning Kaggle Solution"* (equipo ganador, incluye a Chris Deotte), que describe
   explícitamente crear `TransactionAmt` dividido por la media/desviación estándar por tarjeta.
   Aquí se implementan frequency encoding y media de `TransactionAmt` agrupando por `card1`,
   `card2`, `addr1` y por el `UID` del punto 1.
4. **Parte decimal ("cents") de `TransactionAmt`** — mencionado en el mismo blog de NVIDIA y en
   *"IEEE-CIS Fraud Detection" (Abhishek Mishra, Medium)*: el número de decimales / la parte
   decimal de `TransactionAmt` ayuda a separar fraude (los autores incluso señalan que montos con
   3 decimales se relacionan con `addr1`/`addr2` vacíos). Aquí se extrae la parte decimal como
   feature numérica.
5. **Features de `TransactionDT`** — es un contador de segundos desde un origen arbitrario (no una
   fecha real), señalado en la documentación oficial de la competencia y confirmado en el EDA de
   `01_exploracion.ipynb`, donde se vio que la tasa de fraude varía claramente por hora del día. Se
   extraen `hora_del_dia` y `dia_de_semana` vía aritmética modular sobre `TransactionDT`.
6. **Features de dominio de email (`P_emaildomain`, `R_emaildomain`)** — mencionado también en el
   blog de NVIDIA (`P_emaildomain_FE`, frequency encoding del dominio) y en varias soluciones
   públicas: se agrupan dominios de un mismo proveedor (ej. `gmail.com`, `gmail.com.mx` → `gmail`)
   y se crea un flag de coincidencia entre `P_emaildomain` y `R_emaildomain` (comprador vs.
   receptor usan el mismo proveedor de correo).

In [1]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings("ignore")
sys.path.append(str(Path("..").resolve()))
from src.utils import reduce_mem_usage

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
ART_DIR = PROC_DIR / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)


## 0. Carga de datos limpios (de `02_limpieza.ipynb`) y de los emails crudos

In [2]:
X_train = pd.read_parquet(PROC_DIR / "train_clean.parquet")
X_test = pd.read_parquet(PROC_DIR / "test_clean.parquet")

y_train = X_train.pop("isFraud")
y_test = X_test.pop("isFraud")

print("X_train:", X_train.shape, "| X_test:", X_test.shape)


X_train: (472432, 421) | X_test: (118108, 421)


`P_emaildomain` y `R_emaildomain` ya vienen codificadas como enteros (ordinal encoding de
`02_limpieza.ipynb`), lo cual sirve para el modelo pero no permite extraer el "proveedor" del
dominio (ej. agrupar `gmail.com` con `gmail.com.mx`) ni compararlas como texto. Por eso se
recuperan los valores originales de texto desde el CSV crudo (solo esas dos columnas + el
identificador, para no volver a cargar todo el dataset) y se alinean con el split de train/test ya
hecho, usando `TransactionID` como llave.

In [3]:
email_raw = pd.read_csv(
    RAW_DIR / "train_transaction.csv",
    usecols=["TransactionID", "P_emaildomain", "R_emaildomain"],
)

train_emails = X_train[["TransactionID"]].merge(email_raw, on="TransactionID", how="left")
test_emails = X_test[["TransactionID"]].merge(email_raw, on="TransactionID", how="left")
del email_raw

print(train_emails["P_emaildomain"].value_counts(dropna=False).head(10))


P_emaildomain
gmail.com        182612
yahoo.com         80629
NaN               75591
hotmail.com       36283
anonymous.com     29774
aol.com           22572
comcast.net        6215
icloud.com         5015
outlook.com        4141
msn.com            3283
Name: count, dtype: int64


## 1. Features de `TransactionDT`: hora del día y día de la semana

`TransactionDT` es un conteo de segundos desde un origen arbitrario. Con aritmética modular se
puede extraer la hora del día (0-23) y el día de la semana (0-6) sin necesitar una fecha real.

In [4]:
def add_time_features(df):
    df = df.copy()
    df["hora_del_dia"] = ((df["TransactionDT"] // 3600) % 24).astype("int16")
    df["dia_de_semana"] = ((df["TransactionDT"] // 86400) % 7).astype("int16")
    df["dia_absoluto"] = (df["TransactionDT"] // 86400).astype("int32")
    return df

X_train = add_time_features(X_train)
X_test = add_time_features(X_test)
X_train[["hora_del_dia", "dia_de_semana", "dia_absoluto"]].describe()


,hora_del_dia,dia_de_semana,dia_absoluto
count,472432.000000,472432.000000,472432.000000
mean,13.860465,2.958659,84.741777
std,7.608293,2.033950,53.428782
min,0.000000,0.000000,1.000000
25%,6.000000,1.000000,35.000000
50%,16.000000,3.000000,84.000000
75%,20.000000,5.000000,130.000000
max,23.000000,6.000000,182.000000


## 2. Parte decimal de `TransactionAmt`

Se extrae la parte decimal ("cents") del monto como feature numérica separada.

In [5]:
X_train["TransactionAmt_decimal"] = (X_train["TransactionAmt"] - np.floor(X_train["TransactionAmt"])).round(3)
X_test["TransactionAmt_decimal"] = (X_test["TransactionAmt"] - np.floor(X_test["TransactionAmt"])).round(3)
X_train["TransactionAmt_log"] = np.log1p(X_train["TransactionAmt"])
X_test["TransactionAmt_log"] = np.log1p(X_test["TransactionAmt"])


## 3. Feature "UID" (`card1` + `addr1` + `D1n`)

`D1n = día_absoluto - D1` es aproximadamente constante para transacciones del mismo cliente
(ver fuente citada arriba). Se combina con `card1` y `addr1` para aproximar un identificador de
cliente que no existe explícitamente en los datos.

In [6]:
if "D1" in X_train.columns:
    X_train["D1n"] = X_train["dia_absoluto"] - X_train["D1"]
    X_test["D1n"] = X_test["dia_absoluto"] - X_test["D1"]

    X_train["UID"] = (
        X_train["card1"].astype(str) + "_" + X_train["addr1"].astype(str) + "_" + X_train["D1n"].astype(str)
    )
    X_test["UID"] = (
        X_test["card1"].astype(str) + "_" + X_test["addr1"].astype(str) + "_" + X_test["D1n"].astype(str)
    )
    print("UID creado. Clientes únicos aproximados en train:", X_train["UID"].nunique())
else:
    print("D1 no está disponible (se eliminó en 02_limpieza.ipynb por exceso de missing) -> se omite UID.")


UID creado. Clientes únicos aproximados en train: 190612


## 4. Agregaciones por `card1`, `card2`, `addr1` (y `UID` si existe)

Para cada columna de agrupación se calcula, **solo con train**: frecuencia (cuántas transacciones
tiene ese valor) y media de `TransactionAmt`. Ambas tablas se mapean a train y test; los valores
que solo aparecen en test (no vistos en train) quedan como `NaN` y luego se imputan con la media/
frecuencia global de train, que es la forma estándar de evitar fuga de información al mismo tiempo
que se maneja el caso de categorías nuevas.

In [7]:
group_cols = ["card1", "card2", "addr1"]
if "UID" in X_train.columns:
    group_cols.append("UID")

agg_artifacts = {}

for gcol in group_cols:
    freq_map = X_train.groupby(gcol, observed=True).size()
    mean_amt_map = X_train.groupby(gcol, observed=True)["TransactionAmt"].mean()

    global_freq_fill = 0
    global_mean_fill = X_train["TransactionAmt"].mean()

    X_train[f"{gcol}_freq"] = X_train[gcol].map(freq_map).fillna(global_freq_fill).astype("float32")
    X_test[f"{gcol}_freq"] = X_test[gcol].map(freq_map).fillna(global_freq_fill).astype("float32")

    X_train[f"{gcol}_mean_amt"] = X_train[gcol].map(mean_amt_map).fillna(global_mean_fill).astype("float32")
    X_test[f"{gcol}_mean_amt"] = X_test[gcol].map(mean_amt_map).fillna(global_mean_fill).astype("float32")

    agg_artifacts[gcol] = {
        "freq_map": freq_map,
        "mean_amt_map": mean_amt_map,
        "global_freq_fill": global_freq_fill,
        "global_mean_fill": global_mean_fill,
    }

print("Nuevas columnas de agregación:", [c for c in X_train.columns if c.endswith(("_freq", "_mean_amt"))])


Nuevas columnas de agregación: ['card1_freq', 'card1_mean_amt', 'card2_freq', 'card2_mean_amt', 'addr1_freq', 'addr1_mean_amt', 'UID_freq', 'UID_mean_amt']


In [8]:
if "UID" in X_train.columns:
    X_train = X_train.drop(columns=["UID", "D1n"])
    X_test = X_test.drop(columns=["UID", "D1n"])


## 5. Features de dominio de email

Se agrupa cada dominio por su "familia" de proveedor (la parte antes del primer punto, ej.
`gmail.com` y `gmail.com.mx` → `gmail`) y se crea un flag de coincidencia entre el dominio del
comprador (`P_emaildomain`) y del destinatario (`R_emaildomain`). La familia se codifica con
frequency encoding aprendido solo en train.

In [9]:
def email_family(series):
    return series.fillna("missing").str.split(".").str[0]

train_emails["P_family"] = email_family(train_emails["P_emaildomain"])
train_emails["R_family"] = email_family(train_emails["R_emaildomain"])
test_emails["P_family"] = email_family(test_emails["P_emaildomain"])
test_emails["R_family"] = email_family(test_emails["R_emaildomain"])

train_emails["email_match"] = (
    train_emails["P_emaildomain"].notna()
    & (train_emails["P_emaildomain"] == train_emails["R_emaildomain"])
).astype("int8")
test_emails["email_match"] = (
    test_emails["P_emaildomain"].notna()
    & (test_emails["P_emaildomain"] == test_emails["R_emaildomain"])
).astype("int8")

p_family_freq = train_emails["P_family"].value_counts()
r_family_freq = train_emails["R_family"].value_counts()

train_emails["P_family_freq"] = train_emails["P_family"].map(p_family_freq).fillna(0).astype("float32")
test_emails["P_family_freq"] = test_emails["P_family"].map(p_family_freq).fillna(0).astype("float32")
train_emails["R_family_freq"] = train_emails["R_family"].map(r_family_freq).fillna(0).astype("float32")
test_emails["R_family_freq"] = test_emails["R_family"].map(r_family_freq).fillna(0).astype("float32")

email_feature_cols = ["TransactionID", "email_match", "P_family_freq", "R_family_freq"]
X_train = X_train.merge(train_emails[email_feature_cols], on="TransactionID", how="left")
X_test = X_test.merge(test_emails[email_feature_cols], on="TransactionID", how="left")

print(X_train[["email_match", "P_family_freq", "R_family_freq"]].describe())


         email_match  P_family_freq  R_family_freq
count  472432.000000  472432.000000  472432.000000
mean        0.174252  103414.914062  284160.750000
std         0.379326   67507.789062  141783.859375
min         0.000000      26.000000       6.000000
25%         0.000000   36865.000000  362236.000000
50%         0.000000   82219.000000  362236.000000
75%         0.000000  183007.000000  362236.000000
max         1.000000  183007.000000  362236.000000


## 6. Reducción de columnas `V*` por correlación (aprendida solo en train)

Se calcula la matriz de correlación de Pearson de las columnas `V*` restantes **solo en train** y
se elimina una columna de cada par con `|corr| > 0.90` (se conserva la primera de cada grupo muy
correlacionado). La misma lista de columnas eliminadas se aplica a test.

In [10]:
v_cols = [c for c in X_train.columns if c.startswith("V") and c[1:].isdigit()]
print(f"Columnas V disponibles antes de reducir: {len(v_cols)}")

v_cols_to_drop = []
if len(v_cols) > 1:
    corr_matrix = X_train[v_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))
    v_cols_to_drop = [col for col in upper.columns if (upper[col] > 0.90).any()]

    X_train = X_train.drop(columns=v_cols_to_drop)
    X_test = X_test.drop(columns=v_cols_to_drop)

print(f"Columnas V eliminadas por alta correlación (>0.90): {len(v_cols_to_drop)}")
print(f"Columnas V restantes: {len([c for c in X_train.columns if c.startswith('V') and c[1:].isdigit()])}")


Columnas V disponibles antes de reducir: 339


Columnas V eliminadas por alta correlación (>0.90): 161
Columnas V restantes: 178


## 7. Guardado de datasets finales y artefactos para `04_balanceo.ipynb` / `05_modelo.ipynb`

In [11]:
feature_cols = [c for c in X_train.columns if c not in ("TransactionID", "TransactionDT")]

X_train_final = X_train[["TransactionID"] + feature_cols].copy()
X_test_final = X_test[["TransactionID"] + feature_cols].copy()
X_train_final = reduce_mem_usage(X_train_final)
X_test_final = reduce_mem_usage(X_test_final)

X_train_final["isFraud"] = y_train.values
X_test_final["isFraud"] = y_test.values

X_train_final.to_parquet(PROC_DIR / "train_fe.parquet", index=False)
X_test_final.to_parquet(PROC_DIR / "test_fe.parquet", index=False)

joblib.dump(agg_artifacts, ART_DIR / "aggregation_maps.joblib")
joblib.dump(
    {"p_family_freq": p_family_freq, "r_family_freq": r_family_freq},
    ART_DIR / "email_family_freq.joblib",
)
joblib.dump(v_cols_to_drop, ART_DIR / "v_cols_dropped_corr.joblib")

with open(ART_DIR / "feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

print("train_fe.parquet:", X_train_final.shape)
print("test_fe.parquet: ", X_test_final.shape)
print("Total features finales:", len(feature_cols))


Memoria: 453.25 MB -> 451.45 MB (reducción del 0.4%)


Memoria: 113.31 MB -> 112.86 MB (reducción del 0.4%)


train_fe.parquet: (472432, 276)
test_fe.parquet:  (118108, 276)
Total features finales: 274


## 8. Resumen

- **Hora del día / día de semana** desde `TransactionDT` (se elimina la columna cruda
  `TransactionDT` del set de features finales para evitar que el modelo use directamente el
  contador de segundos, que no generaliza).
- **Parte decimal y log1p de `TransactionAmt`**.
- **UID (`card1_addr1_D1n`)** usado solo como columna de agrupación temporal para frequency
  encoding (no se guarda como columna final, ya que es un string de alta cardinalidad).
- **Frequency encoding y media de `TransactionAmt`** por `card1`, `card2`, `addr1` y `UID`.
- **Features de dominio de email**: familia de proveedor (frequency encoding) + flag de
  coincidencia `P_emaildomain == R_emaildomain`.
- **Reducción de columnas `V*`** eliminando las muy correlacionadas entre sí (aprendido solo en
  train), siguiendo el hallazgo de la comunidad de que esto funciona mejor que PCA para este
  dataset.
- Resultado guardado en `data/processed/train_fe.parquet` / `test_fe.parquet`, junto con los
  artefactos necesarios en `data/processed/artifacts/` para que `04_balanceo.ipynb` y
  `05_modelo.ipynb` reutilicen exactamente las mismas columnas de features
  (`artifacts/feature_columns.json`).